# 💻 Notebook do Aluno — Aula 12: Router chains e o conceito de grafo de estado

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 12/14 — Módulo 4: LangGraph e Encerramento**  
**⏱️ 1h40min**  
**🔀 Router Chain · StateGraph mental · LangGraph motivação**  
**🔁 Andaime 50%**  

---

## Como usar este notebook

- Rode as células **na ordem**, de cima para baixo (`Shift+Enter`).
- Complete apenas as partes marcadas com `___` e `👉 LACUNA`.
- Não apague o código já pronto — ele é o andaime da aula.
- Salve sua cópia: **Arquivo > Salvar uma cópia no Drive**.

---

## 🧩 Andaime da aula — complete as lacunas

Complete as lacunas marcadas com `___`.

In [ ]:
!pip install langchain langchain-ollama langchain-chroma pydantic -q

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from pydantic import BaseModel
from typing import Literal
import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
llm        = ChatOllama(model="gpt-oss:120b", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")
retriever  = Chroma(persist_directory="/content/ckp02",
                    embedding_function=embeddings).as_retriever(search_kwargs={"k":3})

# 👉 LACUNA 1: Rota com Literal — adicionar os destinos do domínio do grupo
class Rota(BaseModel):
    destino: Literal[___, ___, ___]  # ex: "rag", "calculadora", "conversa"

# 👉 LACUNA 2: prompt do classificador — descrever cada destino do domínio
prompt_clf = ChatPromptTemplate.from_messages([
    ("system", """___"""),  # descrever cada rota com exemplos do domínio
    ("human", "{input}"),
])
chain_clf = prompt_clf | llm.with_structured_output(Rota)

# Handler RAG (pronto)
handler_rag = ({"contexto": retriever | RunnableLambda(lambda d: "\n".join(x.page_content for x in d)),
                "input": lambda x: x["input"]}
               | ChatPromptTemplate.from_template("Contexto:\n{contexto}\n\nPergunta: {input}")
               | llm | StrOutputParser())

# 👉 LACUNA 3: implementar handler_conversa (chain simples com system prompt amigável)
handler_conversa = (
    ChatPromptTemplate.from_messages([
        ("system", ___),  # persona do assistente do domínio do grupo
        ("human", "{input}"),
    ]) | llm | StrOutputParser()
)

HANDLERS = {"rag": handler_rag, "calculadora": handler_calculadora, "conversa": handler_conversa}

# 👉 LACUNA 4: implementar a função rotear(dados) e montar router_chain
def rotear(dados: dict) -> str:
    rota = chain_clf.invoke({"input": dados[___]})
    print(f"[ROUTER] {rota.destino}")
    return HANDLERS.get(___, HANDLERS["conversa"]).invoke(dados)

router_chain = RunnableLambda(___)

---

## ✍️ Suas anotações

Registre aqui as observações da aula (qualidade dos resultados, comparações e conclusões do grupo).

---

## 🏋️ Exercícios da Aula 12

Prática do Router Chain no domínio do grupo: classificador de intenção com `Literal`, quarta rota no router, fronteiras do classificador e o despacho declarativo que vira `add_conditional_edges` na Aula 13. Rode os andaimes na ordem e complete as lacunas — individual, ~10 min por exercício, direto no Colab.


### Exercício 1 — Classificador de intenção com Literal · ★★☆ · 10 min

*Individual · Colab*

1. Complete o `Literal` do schema com os 3 destinos do router (sugestão: `rag`, `calculadora`, `conversa`).
2. Complete a descrição de cada rota no system prompt, com 1 exemplo de input do domínio.
3. Complete o método do `llm` que força a saída no schema e rode a célula — os 3 testes devem cair nas rotas certas.
4. Ajuste a descrição e rode de novo até os 3 testes acertarem.

> **💡 Dica:** o `Literal` garante o formato da saída; a qualidade do roteamento mora na especificidade da descrição de cada rota no prompt.


In [ ]:
# 👉 LACUNA: classificador de intenção — o andaime do Router (requer `llm` da célula 03)
from pydantic import BaseModel
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate

# 👉 LACUNA 1: complete o Literal com os 3 destinos do router
class RotaEx1(BaseModel):
    destino: Literal[___, ___, ___]

# 👉 LACUNA 2: complete a descrição de cada rota (1 exemplo de input por rota)
prompt_ex1 = ChatPromptTemplate.from_messages([
    ("system", """Você é um roteador de intenções. Classifique o input do usuário
em exatamente um destino:
- rag: perguntas sobre documentos, manuais e contratos do domínio
- calculadora: ___
- conversa: ___

Retorne apenas o JSON com o campo 'destino'."""),
    ("human", "{input}"),
])

# 👉 LACUNA 3: complete o método do llm que força a saída no schema Pydantic
chain_ex1 = prompt_ex1 | llm.___(RotaEx1)

for pergunta in ["Qual a cláusula 5 do contrato?", "Quanto é 450 × 0.88?", "Oi, tudo bem?"]:
    rota = chain_ex1.invoke({"input": pergunta})
    print(f"{pergunta[:35]:35s} → {rota.destino}")


### Exercício 2 — Quarta rota no Router Chain · ★★☆ · 10 min

*Individual · Colab*

1. Complete o `Literal` com a 4ª rota (sugestão: `resumo` — pedidos de síntese dos documentos).
2. Descreva a rota nova no system prompt, com 2 exemplos de input que devem cair nela.
3. Complete o handler do resumo (responde em até 3 linhas, sem opinar) e registre a rota no mapa `HANDLERS_EX`.
4. Complete a função de despacho com fallback e rode com uma pergunta que só faz sentido na rota nova.

> **💡 Dica:** só três pontos mudam — o `Literal`, a descrição no prompt e o registro em `HANDLERS_EX`. O `dict.get` com fallback despacha a rota nova sozinho.


In [ ]:
# 👉 LACUNA: adicionar a rota "resumo" ao Router Chain (requer `llm` da célula 03)
from pydantic import BaseModel
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# 👉 LACUNA 1: 4ª rota no Literal
class RotaEx2(BaseModel):
    destino: Literal["rag", "calculadora", "conversa", ___]

# 👉 LACUNA 2: descreva a rota nova no prompt (2 exemplos de input)
prompt_ex2 = ChatPromptTemplate.from_messages([
    ("system", """Você é um roteador de intenções. Classifique o input do usuário
em exatamente um destino:
- rag: perguntas sobre documentos do domínio
- calculadora: cálculos e conversões numéricas
- conversa: saudações e qualquer outra coisa
- resumo: ___

Retorne apenas o JSON com o campo 'destino'."""),
    ("human", "{input}"),
])
chain_ex2 = prompt_ex2 | llm.with_structured_output(RotaEx2)

# 👉 LACUNA 3: persona do handler do resumo (até 3 linhas, sem opinar)
handler_resumo = (
    ChatPromptTemplate.from_messages([
        ("system", ___),
        ("human", "{input}"),
    ]) | llm | StrOutputParser()
)

# Handlers das rotas base (prontos — respostas curtas para testar o despacho)
HANDLERS_EX = {
    "rag":         RunnableLambda(lambda x: "[RAG] resposta com o contexto do CKP02"),
    "calculadora": RunnableLambda(lambda x: "[CALC] resultado da expressão"),
    "conversa":    RunnableLambda(lambda x: "[CHAT] resposta amigável"),
}
# 👉 LACUNA 4: registre o handler_resumo no mapa
HANDLERS_EX["resumo"] = ___

# 👉 LACUNA 5: chave do despacho no dict.get (com fallback para conversa)
def rotear_ex(dados: dict) -> str:
    rota = chain_ex2.invoke({"input": dados["input"]})
    print(f"[ROUTER] {rota.destino}")
    return HANDLERS_EX.get(___, HANDLERS_EX["conversa"]).invoke(dados)

# 👉 LACUNA 6: pergunta de teste que só faz sentido na rota nova
print(rotear_ex({"input": ___}))


### Exercício 3 — Teste de fronteira com 3 inputs · ★★☆ · 10 min

*Individual · Colab*

1. Complete a célula com 1 input por rota — o 4º teste (caso de fronteira em maiúsculas) já está pronto.
2. Rode a célula 2 vezes com `temperature=0` e observe se alguma rota oscila.
3. Troque o input da rota `rag` por um sinônimo ("manual", "arquivo"...) e rode de novo — o classificador continua acertando?
4. Complete a regra de desempate do caso híbrido (cálculo sobre um documento) que iria no system prompt.

> **💡 Dica:** invoque só o classificador para depurar o roteamento — a decisão errada custa 1 chamada, não 3. O schema garante o formato; a qualidade mora no texto do prompt.


In [ ]:
# 👉 LACUNA: teste de fronteira do classificador (requer `llm` da célula 03)
from pydantic import BaseModel
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate

class RotaEx3(BaseModel):
    destino: Literal["rag", "calculadora", "conversa"]

clf_ex3 = (ChatPromptTemplate.from_messages([
    ("system", """Você é um roteador de intenções. Classifique o input em exatamente um destino:
- rag: perguntas sobre documentos do domínio
- calculadora: cálculos e conversões numéricas
- conversa: saudações e perguntas gerais
Retorne apenas o JSON com o campo 'destino'."""),
    ("human", "{input}"),
]) | llm.with_structured_output(RotaEx3))

# 👉 LACUNA 1, 2 e 3: 1 input por rota — rag, calculadora e conversa
TESTES_EX = [
    ("rag", ___),
    ("calculadora", ___),
    ("conversa", ___),
    ("calculadora", "QUANTO É 450+550?"),   # caso de fronteira (pronto) — pode oscilar
]
for esperado, pergunta in TESTES_EX:
    saidas = [clf_ex3.invoke({"input": pergunta}).destino for _ in range(2)]
    oscilou = " · oscilou" if len(set(saidas)) > 1 else ""
    ok = "✅" if saidas[0] == esperado else "❌"
    print(f"esperado={esperado:12s} obtido={saidas[0]:12s} {ok}{oscilou} | {pergunta[:42]}")

# 👉 LACUNA 4: regra de desempate para o caso híbrido (cálculo sobre documento)
regra_hibrido = ___
print("\nRegra de desempate:", regra_hibrido)


### Exercício 4 — Despacho declarativo — do dict.get ao mapa de rotas · ★★☆ · 10 min

*Individual · Colab*

1. Complete o mapa `MAPA_ROTAS` com a rota que falta (retorno da função → nome do nó destino).
2. Complete `decidir_rota_ex` — o campo do estado que a função de roteamento lê.
3. Preencha o estado de exemplo (input + rota decidida pelo classificador) e rode a simulação.
4. Confira o print final: na Aula 13, esse mapa vira `add_conditional_edges`.

> **💡 Dica:** no grafo, o despacho vira `add_conditional_edges(no_origem, fn_rota, mapa)` — declarado na estrutura, não escondido dentro de um `dict.get`.


In [ ]:
# 👉 LACUNA: do dict.get (Aula 12) ao mapa declarativo (Aula 13) — célula sem LLM

# 👉 LACUNA 1: complete o mapa de despacho com a rota que falta
MAPA_ROTAS = {
    "rag":         "responder_rag",
    "calculadora": "responder_calc",
    ___: ___,
}

# 👉 LACUNA 2: campo do estado que guarda a decisão do classificador
def decidir_rota_ex(estado: dict) -> str:
    return estado[___]

# 👉 LACUNA 3: preencha o estado de exemplo (input + rota decidida)
estado_sim = {"input": ___, "rota": ___}

proximo_no = MAPA_ROTAS.get(decidir_rota_ex(estado_sim), "__end__")
print(f"fn_rota(estado) → {decidir_rota_ex(estado_sim)}")
print(f"Próximo nó: {proximo_no}")
print("Na Aula 13, este mapa vira:")
print('  builder.add_conditional_edges("classificar", decidir_rota_ex,', MAPA_ROTAS, ')')


## 📚 Referências da aula

- Docs LangChain — RunnableLambda e routing patterns. Como construir chains com branching usando RunnableLambda e with_structured_output. python.langchain.com/docs/how_to/routing
- Docs LangGraph — Conceitos de StateGraph, nodes e edges. A leitura recomendada antes da Aula 13. langchain-ai.github.io/langgraph/concepts/low_level
- Blog Anthropic Engineering — "Building Effective Agents" (2025). Seção sobre orchestrators e subagents — a motivação arquitetural para grafos de estado. anthropic.com/engineering/building-effective-agents
- Livro Russell, S.; Norvig, P. — Inteligência Artificial. 3ª ed. Pearson, 2016. Cap. 3 — Resolução de problemas como busca: a teoria por trás de grafos de estado em IA, base conceitual do LangGraph.
- Livro Polzer, D. — RAG with Python Cookbook. O'Reilly, 2026. O custo de latência de um router baseado em LLM (500ms-2s) e a alternativa de classificador leve sobre embeddings — a fundamentação por trás do classificador de intenção desta aula.
- Livro Gullí, A. — Agentic Design Patterns. O'Reilly, 2025. Cap. 3 — Routing: as três formas de implementar roteamento (regras, classificador de ML, LLM) e o trade-off de custo/latência por trás do Router Chain desta aula.

---

**Próxima Aula — Aula 13** — LangGraph — StateGraph, conditional edges e HITL
  
O diagrama desta aula vira código. Estado TypedDict, add_node(), add_conditional_edges(), MemorySaver, interrupt_before.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*